In [ ]:
import sympy as sp
from IPython.display import display

# ==============================================================================
# 1. SymPy Settings and Symbol Definitions
# ==============================================================================

sp.init_printing(use_unicode=True)

z = sp.Symbol('z', complex=True)
y = sp.Symbol('y')
alpha = sp.Symbol('alpha', real=True)
n = sp.Symbol('n', integer=True, nonnegative=True)


# ==============================================================================
# 2. METHOD: Partial Fraction Expansion Method
# ==============================================================================

print("=== METHOD: Partial Fraction Expansion Method ===")

X_y = (1 - alpha * y) / (y - alpha)

print("\n1. Function X(y) where y = z^(-1):")
display(X_y)

pfe_y = sp.apart(X_y, y)

print("\n2. Partial Fraction Expansion:")
display(pfe_y)

def table_lookup_inverse_z(term, y_var, n_var):
    u = sp.Heaviside(n_var)
    delta = sp.KroneckerDelta(n_var, 0)
    
    # If the term is a pure constant (no y dependency)
    if not term.has(y_var):
        return sp.simplify(term * delta)
        
    _, den = sp.fraction(sp.cancel(term))
    
    for base, power in sp.factor_list(den)[1]:
        roots = sp.solve(base, y_var)
        if not roots:
            continue
        
        y0 = roots[0]
        a = sp.simplify(1 / y0)
        scale = sp.simplify(-sp.Poly(base, y_var).LC() * y0)
        coeff = sp.simplify(sp.limit(term * base**power, y_var, y0) / scale**power)
        
        if power == 1:
            return sp.simplify(coeff * a**n_var * u)
            
    return term

def inverse_z_transform(expr, y_var, n_var):
    expr = sp.expand(expr)
    if isinstance(expr, sp.Add):
        return sp.simplify(sum(inverse_z_transform(t, y_var, n_var) for t in expr.args))
    return table_lookup_inverse_z(expr, y_var, n_var)

x_n = inverse_z_transform(pfe_y, y, n)

print("\n3. Final inverse Z-transform signal x[n]:")
display(x_n)